# 用 Qwen3-TTS 为 tiny-voca-mix 设计音色并朗读全量语料
在 Colab 选择 **GPU**，从上至下运行。第一格可以修改人物声音描述，默认是一位具有央视新闻播报风格的男播音员。VoiceDesign 只设计一次，Base 用保存的参考音朗读中文、英文和中英混读句子。参考音和批量合成分开运行，两套模型依次占用显存。[Qwen3-TTS 官方用法](https://github.com/QwenLM/Qwen3-TTS)

工作目录是 **我的云端硬盘 / tiny-voca-mix-qwen3-tts**。输入清单为 `inputs/corpus.jsonl`，共 31,799 句。模型下载到 `models/`，参考音和设计参数在 `voices/`，合成结果在 `runs/`。请为两套模型和全量音频预留约 20 GB 可用空间，实际用量取决于生成时长。

**首次试听**可勾选 `PREVIEW_ONLY`，每类先生成一句。确认音色后取消勾选，从头运行即可复用这三句。**断线续跑**时按原设置重新执行，程序逐条检查已落盘的 WAV，补生成缺失或截断的文件。调节 `CONCURRENCY` 可以改变单 GPU 同一批次并行生成的句数；遇到显存不足会减半重试。请在一个 Colab 会话中运行同一任务。


In [ ]:
#@title 挂载 Drive，设置音色和并发
from google.colab import drive
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import subprocess
import sys

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/tiny-voca-mix-qwen3-tts")
VOICE_DESCRIPTION = "一位成熟的男新闻播音员，音色沉稳浑厚，中低音区饱满。使用标准普通话，吐字清楚，停连自然，语速平稳，有央视新闻播报的沉着感。切换到英语时保持同一音色，重音自然。" #@param {type:"string"}
REFERENCE_TEXT = "各位听众，早上好。今天上午，连接两岸的新桥正式通车。 The new route will save commuters about twenty minutes." #@param {type:"string"}
VOICE_SEED = 42 #@param {type:"integer"}
CONCURRENCY = 2 #@param {type:"integer"}
PREVIEW_ONLY = False #@param {type:"boolean"}

if type(CONCURRENCY) is not int or CONCURRENCY < 1:
    raise ValueError("CONCURRENCY 必须是正整数")
if not VOICE_DESCRIPTION.strip() or not REFERENCE_TEXT.strip() or not 0 <= VOICE_SEED < 2**32:
    raise ValueError("请输入音色描述、参考台词，以及 0 到 2**32-1 之间的随机种子。")

CORPUS = ROOT / "inputs/corpus.jsonl"
if not CORPUS.is_file():
    raise FileNotFoundError(f"请确认已将文本清单上传到 {CORPUS}")
def digest(data):
    return hashlib.sha256(data).hexdigest()

raw = CORPUS.read_bytes()
rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
language_map = {"zh": "Chinese", "en": "English", "mixed": "Auto"}
if not rows or len({r["id"] for r in rows}) != len(rows):
    raise ValueError("清单为空或存在重复句子 ID")
import re
for row in rows:
    if (not re.fullmatch(r"[A-Za-z0-9_-]+", row["id"])
        or row["language"] not in language_map or not row["text"].strip()
        or row["split"] not in {"train", "validation", "test"}):
        raise ValueError(f"无效的清单条目 {row['id']}")
corpus_sha = digest(raw)
print(f"{len(rows):,} 条", dict(Counter(r["language"] for r in rows)))
for lang in language_map:
    example = next((r["text"] for r in rows if r["language"] == lang), None)
    if example:
        print(lang, example)

def atomic_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)

voice = {
    "description": VOICE_DESCRIPTION, "reference_text": REFERENCE_TEXT, "reference_language": "Auto", "seed": VOICE_SEED,
    "design_revision": "5ecdb67327fd37bb2e042aab12ff7391903235d3", "qwen_tts": "0.1.1",
}
voice_json = json.dumps(voice, ensure_ascii=False, sort_keys=True, indent=2)
VOICE_DIR = ROOT / "voices" / digest(voice_json.encode())[:16]
atomic_text(VOICE_DIR / "design.json", voice_json)
REFERENCE = VOICE_DIR / "reference.wav"
BASE_REVISION = "fd4b254389122332181a7c3db7f27e918eec64e3"
WORK = Path("/content/tiny-voca-qwen-runtime")
WORK.mkdir(exist_ok=True)


## 安装推理依赖
Python 环境装在 Colab 临时盘，可在新会话中重建。权重、下载缓存和合成数据均写入 Drive。


In [ ]:
#@title 配置 GPU 推理环境
import shutil
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi", "-L"]).returncode:
    raise RuntimeError("请在 Colab 的「代码执行程序 → 更改运行时类型」中选择 GPU，再运行本单元。")

env = os.environ.copy()
env.update(HF_HOME=str(ROOT / "cache/huggingface"), TORCH_HOME=str(ROOT / "cache/torch"),
           HF_HUB_DISABLE_XET="1", TOKENIZERS_PARALLELISM="false")
def run(*args):
    subprocess.run([str(a) for a in args], check=True, env=env)

if not shutil.which("sox"):
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-qy", "sox")
run(sys.executable, "-m", "pip", "install", "-q", "uv")
PYTHON = WORK / "venv/bin/python"
if not PYTHON.exists():
    run("uv", "venv", "--python", "3.11", WORK / "venv")
run("uv", "pip", "install", "--python", PYTHON, "--index-url", "https://download.pytorch.org/whl/cu126",
    "torch==2.8.0", "torchaudio==2.8.0")
run("uv", "pip", "install", "--python", PYTHON, "qwen-tts==0.1.1", "gradio==5.49.1", "numpy==2.2.6")
run(PYTHON, "-c", "import torch, qwen_tts; assert torch.cuda.is_available(), 'PyTorch 未检测到 GPU'; print(torch.cuda.get_device_name())")


## 准备推理程序
下面的单元把程序写入临时运行环境。输出 WAV 按语言和句子 ID 分目录，音频写完并验证后才使用正式文件名。


In [ ]:
%%writefile /content/tiny-voca-qwen-runtime/synthesize.py
import argparse
from collections import Counter
import gc
import hashlib
import json
import os
from pathlib import Path
import wave

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import snapshot_download
from qwen_tts import Qwen3TTSModel
from tqdm import tqdm

RATE = 24000
LANGUAGE = {"zh": "Chinese", "en": "English", "mixed": "Auto"}


def load_model(root, kind, revision):
    name = f"Qwen3-TTS-12Hz-1.7B-{kind}"
    folder = root / "models" / name / revision
    # local_dir 把权重、分词器和可恢复的未完成下载一起放在 Drive。
    snapshot_download("Qwen/" + name, revision=revision, local_dir=folder,
                      allow_patterns=["*.json", "*.txt", "*.safetensors"], max_workers=2)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return Qwen3TTSModel.from_pretrained(str(folder), device_map="cuda:0", dtype=dtype,
                                         attn_implementation="sdpa", local_files_only=True)


def wav_frames(path):
    # 验证 PCM 头和最后一帧，防止断写文件被跳过。
    try:
        with wave.open(str(path), "rb") as handle:
            frames = handle.getnframes()
            if (frames < 1 or handle.getframerate() != RATE or handle.getnchannels() != 1
                or handle.getsampwidth() != 2):
                return 0
            handle.setpos(frames - 1)
            return frames if len(handle.readframes(1)) == 2 else 0
    except (FileNotFoundError, wave.Error, EOFError):
        return 0


def save_wav(path, samples, rate):
    samples = np.asarray(samples)
    if rate != RATE or samples.ndim != 1 or not samples.size or not np.isfinite(samples).all():
        raise ValueError(f"无效的模型输出 {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".wav.part")
    sf.write(tmp, samples, rate, format="WAV", subtype="PCM_16")
    if wav_frames(tmp) != samples.size:
        raise IOError(f"WAV 写入不完整 {tmp}")
    os.replace(tmp, path)


def design_voice(root, voice_dir):
    ref = voice_dir / "reference.wav"
    if wav_frames(ref):
        print("使用已保存的参考音", ref)
        return
    config = json.loads((voice_dir / "design.json").read_text())
    torch.manual_seed(config["seed"])
    model = load_model(root, "VoiceDesign", config["design_revision"])
    wavs, rate = model.generate_voice_design(text=config["reference_text"], language=config["reference_language"],
                                            instruct=config["description"])
    save_wav(ref, wavs[0], rate)


def audio_path(row):
    bucket = hashlib.sha256(row["id"].encode()).hexdigest()[:2]
    return Path("audio") / row["language"] / bucket / (row["id"] + ".wav")


def synthesize(root, voice_dir, out, revision, corpus_sha, concurrency, preview):
    raw = (root / "inputs/corpus.jsonl").read_bytes()
    if hashlib.sha256(raw).hexdigest() != corpus_sha:
        raise ValueError("语料已变更，请从配置单元重新运行。")
    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if preview:
        rows = [next(r for r in rows if r["language"] == lang)
                for lang in LANGUAGE if any(r["language"] == lang for r in rows)]
    paths = {r["id"]: audio_path(r) for r in rows}
    completed = {}
    pending = []
    for row in tqdm(rows, desc="检查 Drive 音频"):
        frames = wav_frames(out / paths[row["id"]])
        if frames:
            completed[row["id"]] = frames
        else:
            pending.append(row)

    print(f"本次目标 {len(rows)} 条，复用 {len(completed)} 条，待生成 {len(pending)} 条。", flush=True)
    if pending:
        model = load_model(root, "Base", revision)
        config = json.loads((voice_dir / "design.json").read_text())
        prompt = model.create_voice_clone_prompt(ref_audio=str(voice_dir / "reference.wav"),
                                                ref_text=config["reference_text"])
        # 长度相近的句子放在同一个 GPU batch，减少等待最长句的时间。
        pending.sort(key=lambda r: len(r["text"]))
        offset, batch_size = 0, concurrency
        with tqdm(total=len(rows), initial=len(completed), desc="合成") as bar:
            while offset < len(pending):
                batch = pending[offset:offset + batch_size]
                try:
                    wavs, rate = model.generate_voice_clone(
                        text=[r["text"] for r in batch], language=[LANGUAGE[r["language"]] for r in batch],
                        voice_clone_prompt=prompt)
                except torch.cuda.OutOfMemoryError:
                    if len(batch) == 1:
                        raise
                    batch_size = max(1, len(batch) // 2)
                    print(f"显存不足，并发降至 {batch_size}，重试当前批。", flush=True)
                else:
                    if len(wavs) != len(batch):
                        raise RuntimeError("模型返回的音频数量与输入不符")
                    for row, samples in zip(batch, wavs):
                        save_wav(out / paths[row["id"]], samples, rate)
                        completed[row["id"]] = len(samples)
                        bar.update()
                    offset += len(batch)
                    continue
                gc.collect()
                torch.cuda.empty_cache()

    manifest = out / ("preview.jsonl" if preview else "corpus.jsonl")
    tmp = manifest.with_suffix(".jsonl.part")
    with tmp.open("w", encoding="utf-8") as stream:
        for row in rows:
            record = {**row, "audio": paths[row["id"]].as_posix(), "voice_id": voice_dir.name,
                      "sample_rate": RATE, "seconds": completed[row["id"]] / RATE}
            print(json.dumps(record, ensure_ascii=False), file=stream)
    os.replace(tmp, manifest)
    print("已完成", dict(Counter(r["language"] for r in rows)), manifest)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("stage", choices=["design", "synthesize"])
    parser.add_argument("root", type=Path)
    parser.add_argument("voice_dir", type=Path)
    parser.add_argument("--out", type=Path)
    parser.add_argument("--revision")
    parser.add_argument("--corpus-sha")
    parser.add_argument("--concurrency", type=int, default=2)
    parser.add_argument("--preview", action="store_true")
    args = parser.parse_args()
    if args.stage == "design":
        design_voice(args.root, args.voice_dir)
    else:
        synthesize(args.root, args.voice_dir, args.out, args.revision, args.corpus_sha, args.concurrency, args.preview)


## 设计并试听参考音
调整音色描述、参考台词或种子会创建新音色目录。点击播放器即可试听，然后运行下一节。


In [ ]:
from IPython.display import Audio, display
WORKER = WORK / "synthesize.py"
run(PYTHON, "-u", WORKER, "design", ROOT, VOICE_DIR)
display(Audio(filename=str(REFERENCE)))


## 生成语料并导出清单
中文使用 Chinese，英文使用 English，混读句保持整句并使用 Auto。每条 WAV 为 24 kHz、16 位单声道。完整运行会写出带原始 ID、文本、划分、来源、相对音频路径和时长的 `corpus.jsonl`。试听模式写入 `preview.jsonl`。


In [ ]:
reference_sha = digest(REFERENCE.read_bytes())
recipe = {"corpus_sha256": corpus_sha, "reference_sha256": reference_sha, "voice": VOICE_DIR.name,
          "base_revision": BASE_REVISION, "language_map": language_map, "format": "pcm16-24k-v1"}
recipe_json = json.dumps(recipe, ensure_ascii=False, indent=2, sort_keys=True)
OUT = ROOT / "runs" / digest(recipe_json.encode())[:16]
atomic_text(OUT / "recipe.json", recipe_json)
print("本次输出目录", OUT, flush=True)
run(PYTHON, "-u", WORKER, "synthesize", ROOT, VOICE_DIR, "--revision", BASE_REVISION,
    "--corpus-sha", corpus_sha, "--out", OUT, "--concurrency", CONCURRENCY,
    *(["--preview"] if PREVIEW_ONLY else []))


## 试听三类输出


In [ ]:
manifest = OUT / ("preview.jsonl" if PREVIEW_ONLY else "corpus.jsonl")
samples = {}
with manifest.open(encoding="utf-8") as stream:
    for line in stream:
        record = json.loads(line)
        samples.setdefault(record["language"], record)
        if len(samples) == len(language_map):
            break
for language, record in samples.items():
    print(language, record["text"])
    display(Audio(filename=str(OUT / record["audio"])))
print("输出清单", manifest)
